# Clean the dataset
Kode ini untuk membersihkan file-file yang tidak sama. Misalnya folder image_dir: 5972, sedangkan semantic_dir: 5059. Sekalian menyeleksi isi summary.csv

## Preparing

In [1]:
import csv
import os
import shutil
from pathlib import Path

In [17]:
def sinkronisasi_dataset_dengan_csv(base_dir, output_dir):
    base_path = Path(base_dir)
    output_path = Path(output_dir)
    
    # Path folder sumber dan file CSV asli
    img_left_dir    = base_path / 'img_left'
    disparity_dir   = base_path / 'disparity'
    semantic_dir    = base_path / 'semantic'
    landmark_dir    = base_path / 'landmark'
    vis_2d_landmark = base_path / 'vis_2d_landmark'
    csv_asal        = base_path / 'pseudo_ground_truth_summary.csv'
    
    # Path folder dan file CSV hasil filter
    out_img_left    = output_path / 'img_left'
    out_semantic    = output_path / 'semantic'
    out_disparity   = output_path / 'disparity'
    out_landmark    = output_path / 'landmark'
    out_vis_2d      = output_path / 'vis_2d_landmark'
    csv_tujuan      = output_path / 'summary_filtered.csv'
    
    # Validasi awal keberadaan folder/file penting
    if not img_left_dir.exists() or not semantic_dir.exists():
        print("Error: Folder 'img_left' / 'semantic' tidak ditemukan!")
        return
    if not disparity_dir.exists() or not vis_2d_landmark.exists() or not landmark_dir.exists():
        print("Error: Folder 'disparity' / 'vis_2d_landmark' / 'landmark' tidak ditemukan!")
        return
    if not csv_asal.exists():
        print("Error: File 'summary.csv' tidak ditemukan di folder dataset!")
        return

    # Buat folder output baru
    out_img_left.mkdir(parents=True, exist_ok=True)
    out_semantic.mkdir(parents=True, exist_ok=True)
    out_disparity.mkdir(parents=True, exist_ok=True)
    out_landmark.mkdir(parents=True, exist_ok=True)
    out_vis_2d.mkdir(parents=True, exist_ok=True)

    print("Membaca file gambar...")
    img_left_files          = {f.stem: f for f in img_left_dir.glob('*.png')}
    semantic_files          = {f.stem: f for f in semantic_dir.glob('*.png')}
    disparity_files         = {f.stem: f for f in disparity_dir.glob('*.png')}
    landmark_files          = {f.stem: f for f in landmark_dir.glob('*.csv')}
    vis_2d_landmark_files   = {f.stem: f for f in vis_2d_landmark.glob('*.png')}
    
    
    # Cari nama file (tanpa ekstensi) yang ada di KEDUA folder gambar
    nama_cocok = set(img_left_files.keys()).intersection(set(semantic_files.keys()))
    
    print(f"Total gambar di img_left: {len(img_left_files)}")
    print(f"Total gambar di semantic: {len(semantic_files)}")
    print(f"Jumlah gambar berpasangan: {len(nama_cocok)}")
    print("-" * 50)
    print("Memproses penyaringan CSV dan penyalinan gambar...")

    baris_terfilter = 0
    
    # Membuka file CSV asal dan membuat file CSV tujuan baru
    with open(csv_asal, mode='r', encoding='utf-8', newline='') as f_in, \
         open(csv_tujuan, mode='w', encoding='utf-8', newline='') as f_out:
        
        reader = csv.reader(f_in)
        writer = csv.writer(f_out)
        
        # Ambil baris pertama sebagai header kolom
        header = next(reader)
        writer.writerow(header)
        
        # Cari indeks kolom 'filename' secara otomatis
        try:
            filename_idx = header.index('filename')
        except ValueError:
            print("Error: Kolom 'filename' tidak ditemukan di summary.csv!")
            return

        # Iterasi baris data CSV satu per satu
        for row in reader:
            if not row:  # Lewati jika ada baris kosong
                continue
                
            # Ambil nilai kolom filename, lalu ubah ke nama dasarnya (tanpa .jpg)
            file_csv = row[filename_idx]
            stem_csv = Path(file_csv).stem
            
            # CEK: Apakah nama file di CSV ini termasuk dalam gambar yang berpasangan?
            if stem_csv in nama_cocok:
                # 1. Tulis baris data ke CSV baru
                writer.writerow(row)
                baris_terfilter += 1
                
                # 2. Salin gambar JPG yang relevan
                src_jpg = img_left_files[stem_csv]
                dst_jpg = out_img_left / src_jpg.name
                if not dst_jpg.exists(): # Mencegah duplikasi salin jika file CSV punya nama ganda
                    shutil.copy2(src_jpg, dst_jpg)
                
                # 3. Salin gambar PNG yang relevan
                src_png = semantic_files[stem_csv]
                dst_png = out_semantic / src_png.name
                if not dst_png.exists():
                    shutil.copy2(src_png, dst_png)

    print("-" * 50)
    print("Selesai!")
    print(f"-> Gambar tersaring disalin ke: {output_dir}")
    print(f"-> File CSV baru disimpan di : {csv_tujuan}")
    print(f"-> Total data yang lolos seleksi: {baris_terfilter} baris data.")


## Do it!

In [18]:
%pwd

'/home/dionsetiawan/Documents/Riset/Progress-Tesis/thesis_ws/HaOri-6D/yon-poseCNN/data-extraction'

In [19]:

# --- KONFIGURASI PATH ---
PATH_DATA_EXTRACTION = "./pseudo_dataset_08072026" 
PATH_HASIL_SELEKSI = "./pseudo_dataset_08072026/dataset_filtered"

# Jalankan fungsi
sinkronisasi_dataset_dengan_csv(PATH_DATA_EXTRACTION, PATH_HASIL_SELEKSI)


Membaca file gambar...
Total gambar di img_left: 5972
Total gambar di semantic: 5059
Jumlah gambar berpasangan: 0
--------------------------------------------------
Memproses penyaringan CSV dan penyalinan gambar...
--------------------------------------------------
Selesai!
-> Gambar tersaring disalin ke: ./pseudo_dataset_08072026/dataset_filtered
-> File CSV baru disimpan di : pseudo_dataset_08072026/dataset_filtered/summary_filtered.csv
-> Total data yang lolos seleksi: 0 baris data.
